# 00 — Data: what / how / where / use

Every dataset is one entry in `configs/datasets.yaml`. This notebook walks the four questions and ends with a POE-ready panel.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path("..").resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from sp500rl.config import load_config
from sp500rl.seed import set_seed

CFG = load_config(ROOT / "configs" / "default.yaml")
SEED = set_seed(int(CFG["seed"]))
DATASET = CFG["dataset"]            # configs/datasets.yaml entry (synthetic | ab_wrds | yahoo | ...)
UNIVERSE = CFG["universe"]["rule"]  # ab_finrl | sandbox | full_window | top_n | listed
print(f"seed={SEED} dataset={DATASET} universe={UNIVERSE}")
print("train", CFG["dates"]["train_start"], "→", CFG["dates"]["train_end"])
print("test ", CFG["dates"]["test_start"], "→", CFG["dates"]["test_end"])


## 1. What datasets exist?

In [ ]:
from sp500rl.data import list_datasets, status, fetch, load_prices, get_panel, DatasetNotReady

for ds in list_datasets():
    print(f"{ds.name:<10} how={ds.how:<9} format={ds.format:<9} {ds.what}")


## 2. Where do the files go, and are they there?

In [ ]:
for name, info in status().items():
    print(name, "ready" if info["ready"] else "MISSING")
    for role, f in info["files"].items():
        print("   ", "ok " if f["exists"] else "-- ", role, f["path"])


## 3. How do I retrieve one?

`fetch` generates / downloads when it can. A manual dataset (the AB_finRL extract) raises with the instruction and the exact paths.

In [ ]:
written = fetch(DATASET, CFG)
print("files:", {k: str(v) for k, v in written.items()})

try:
    fetch("ab_wrds", CFG)
except DatasetNotReady as exc:
    print(exc)


## 4. Where do I use it?

`load_prices` → canonical `date, tic, open, high, low, close, volume`. `get_panel` → universe → features → balanced POE panel, cached in `data/processed/<dataset>__<universe>.parquet`.

In [ ]:
import matplotlib.pyplot as plt
from sp500rl.data.schema import validate
from sp500rl.data.panel import assert_balanced_panel

prices = load_prices(DATASET, CFG)
print(validate(prices).summary())
print("tickers", sorted(prices["tic"].unique()))

panel = get_panel(DATASET, UNIVERSE, CFG, refresh=True)
assert_balanced_panel(panel, feature_cols=CFG["poe"]["features"])
print("panel", panel.shape, panel["date"].min().date(), "→", panel["date"].max().date())

pivot = panel.pivot(index="date", columns="tic", values="close")
ax = pivot.iloc[:, :4].plot(figsize=(10, 4), title=f"{DATASET} close (first 4 tickers)")
ax.set_ylabel("close")
plt.tight_layout()
plt.show()


## Same code, other datasets

Switch `dataset:` in `configs/default.yaml` (or pass a name). Below: the AB_finRL-shaped path with fake files so it runs before the real CSVs exist.

In [ ]:
fetch("ab_wrds", CFG, fake=True)   # writes CRSP-shaped wrds_processed / wrds_tickers
ab_panel = get_panel("ab_wrds", "ab_finrl", CFG, refresh=True)
print("ab_wrds panel", ab_panel.shape, sorted(ab_panel["tic"].unique()))
assert set(ab_panel["tic"]) == set(panel["tic"])
